In [33]:
import yfinance
import pandas as pd




In [ ]:
holdings = {
    "VWRP.L": "Global equity",      # Vanguard FTSE All-World (Acc)
    "XMMS.L": "Emerging markets",   # Xtrackers MSCI EM 1C, GBP line
    "EQGB.L": "Nasdaq-100",         # Invesco Nasdaq-100 
    "LGEN.L": "Legal & General",
    "SEMI.L": "Semiconductors",     # iShares MSCI Global Semiconductors (Acc)
    "SGLN.L": "Gold",               # iShares Physical Gold
}


for ticker in holdings:
    g = yfinance.Ticker(ticker)
    print(g.info["longName"])




Vanguard FTSE All-World UCITS ETF USD Accumulation
Xtrackers MSCI Emerging Markets UCITS ETF 1C
Invesco EQQQ NASDAQ-100 UCITS ETF (GBP Hdg)
Legal & General Group Plc
iShares MSCI Global Semiconductors UCITS ETF USD Acc
iShares Physical Gold ETC


In [29]:
raw_prices = yfinance.download(list(holdings), period="3y", interval="1mo", auto_adjust=True)["Close"]

[*********************100%***********************]  6 of 6 completed


In [30]:

prices = raw_prices.iloc[:-1]
prices = prices.rename(columns=holdings)

print(prices.isna().sum())


Ticker
Nasdaq-100          0
Legal & General     0
Semiconductors      0
Gold                0
Global equity       0
Emerging markets    0
dtype: int64


In [31]:
returns = prices.pct_change().dropna()
print(returns.head())

Ticker      Nasdaq-100  Legal & General  Semiconductors      Gold  \
Date                                                                
2023-12-01    0.066837         0.095072        0.108384  0.006378   
2024-01-01    0.018512         0.015532       -0.194737 -0.005545   
2024-02-01    0.041777        -0.050588        0.111653  0.003346   
2024-03-01    0.016610         0.050805        0.049956  0.084789   
2024-04-01   -0.032446        -0.071781       -0.043398  0.045082   

Ticker      Global equity  Emerging markets  
Date                                         
2023-12-01       0.045209          0.032612  
2024-01-01       0.011136         -0.038202  
2024-02-01       0.039638          0.044708  
2024-03-01       0.034596          0.031734  
2024-04-01      -0.018698          0.012186  


In [32]:
returns.describe()

Ticker,Nasdaq-100,Legal & General,Semiconductors,Gold,Global equity,Emerging markets
count,34.000000,34.000000,34.000000,34.000000,34.000000,34.000000
mean,0.020812,0.008285,0.037722,0.021132,0.015079,0.017825
std,0.052659,0.046938,0.122227,0.055774,0.033412,0.045859
min,-0.071397,-0.092149,-0.225136,-0.105599,-0.060334,-0.101663
25%,-0.005060,-0.030969,-0.040990,-0.005992,-0.004859,-0.010807
50%,0.018587,0.013875,0.033814,0.009225,0.010256,0.011934
75%,0.048576,0.047365,0.113372,0.058766,0.042271,0.048714
max,0.166780,0.095072,0.360510,0.134792,0.075368,0.117840


In [ ]:
annual_return = returns.mean() * 12 #Multiplied by 12 to get annual instead of monthly
print(annual_return)

Ticker
Nasdaq-100          0.249745
Legal & General     0.099418
Semiconductors      0.452661
Gold                0.253581
Global equity       0.180945
Emerging markets    0.213898
dtype: float64


In [ ]:
annual_vol = returns.std() * 12 ** 0.5  #Multipled by 12 then square rooted to get annual instead of monthly
print(annual_vol)

Ticker
Nasdaq-100          0.182416
Legal & General     0.162597
Semiconductors      0.423406
Gold                0.193206
Global equity       0.115741
Emerging markets    0.158859
dtype: float64


In [42]:
summary = pd.DataFrame({"annual_return": annual_return, "annual_vol": annual_vol})
print(summary)

                  annual_return  annual_vol
Ticker                                     
Nasdaq-100             0.249745    0.182416
Legal & General        0.099418    0.162597
Semiconductors         0.452661    0.423406
Gold                   0.253581    0.193206
Global equity          0.180945    0.115741
Emerging markets       0.213898    0.158859


In [ ]:

weights = pd.Series({"Global equity": 0.3, "Emerging markets": 0.2, "Nasdaq-100": 0.15, "Legal & General": 0.15, "Semiconductors": 0.1, "Gold": 0.1})
print(weights)
print(weights.sum()) #Quick check that weights add to 1



Global equity       0.30
Emerging markets    0.20
Nasdaq-100          0.15
Legal & General     0.15
Semiconductors      0.10
Gold                0.10
dtype: float64
1.0


In [46]:
portfolio_returns = returns @ weights
print(portfolio_returns.head())

Date
2023-12-01    0.055848
2024-01-01   -0.019221
2024-02-01    0.031011
2024-03-01    0.040312
2024-04-01   -0.018638
dtype: float64


In [ ]:
#Annualising weighted portfolio returns
portfolio_annual_return = portfolio_returns.mean() * 12 #Multiplied by 12 to get annual instead of monthly
print(portfolio_annual_return)

portfolio_annual_vol = portfolio_returns.std() * 12 ** 0.5  #Multipled by 12 then square rooted to get annual instead of monthly
print(portfolio_annual_vol)


0.22006179666273978
0.13642123899483316


In [ ]:
summary.loc["Portfolio"] = [portfolio_annual_return, portfolio_annual_vol] #Adds a new row to existing summary of weighted portfolio
print(summary)

                  annual_return  annual_vol
Ticker                                     
Nasdaq-100             0.249745    0.182416
Legal & General        0.099418    0.162597
Semiconductors         0.452661    0.423406
Gold                   0.253581    0.193206
Global equity          0.180945    0.115741
Emerging markets       0.213898    0.158859
Portfolio              0.220062    0.136421


In [59]:
print((annual_vol * weights).sum())

0.17990731785002778


## Finding 1: Diversification cuts my risk by about a quarter
- If my holdings' risks simply averaged out, my portfolio's volatility would be 18.0%.
- The actual figure is 13.6%, which is 4.4 percentage points (about 24%) lower.
- The return doesn't drop at all. It stays at 22.0%, exactly the weighted average of my holdings.
- Returns average out but risk doesn't, because my holdings don't all fall at the same time.


In [ ]:
#Sharpe ratio

boe_interest_rate = 0.0375 #bank of england interest rate as at 14/10/2026

summary["sharpe"] = (summary["annual_return"] - boe_interest_rate)/(summary["annual_vol"])

print(summary["sharpe"])


Ticker
Nasdaq-100          1.163524
Legal & General     0.380806
Semiconductors      0.980526
Gold                1.118398
Global equity       1.239360
Emerging markets    1.110406
Portfolio           1.338221
Name: sharpe, dtype: float64


## Finding 2: My portfolio beats every holding on risk-adjusted return
- My portfolio's Sharpe ratio is 1.34. The best single holding is Global equity at 1.24 and the worst is Legal & General at 0.38.
- Semiconductors had by far the highest return (45.3% a year) but a Sharpe ratio of only 0.98, because its volatility was 42.3%.
- Looking at return on its own would have pointed me to the wrong holding.
- These three years were very strong for markets, so the figures describe the past, not the future.



In [68]:
#Correlation matrix

corr = returns.corr().round(2)
print(corr)

Ticker            Nasdaq-100  Legal & General  Semiconductors  Gold  \
Ticker                                                                
Nasdaq-100              1.00             0.21            0.80 -0.02   
Legal & General         0.21             1.00            0.20 -0.08   
Semiconductors          0.80             0.20            1.00  0.01   
Gold                   -0.02            -0.08            0.01  1.00   
Global equity           0.85             0.29            0.72  0.17   
Emerging markets        0.72             0.17            0.84  0.38   

Ticker            Global equity  Emerging markets  
Ticker                                             
Nasdaq-100                 0.85              0.72  
Legal & General            0.29              0.17  
Semiconductors             0.72              0.84  
Gold                       0.17              0.38  
Global equity              1.00              0.78  
Emerging markets           0.78              1.00  


## Finding 3: My four equity funds behave like one big bet
- Global equity, Emerging markets, Nasdaq-100 and Semiconductors make up 75% of my portfolio, and the correlations between them range from 0.72 to 0.85.
- Nasdaq-100 and Global equity (0.85) both hold the same large US tech companies.
- Semiconductors and Emerging markets (0.84): emerging-market indices hold big Asian chipmakers, so part of my EM fund is really another semiconductor bet.
- The real diversifiers are Gold (close to zero with most holdings, -0.08 to 0.17) and Legal & General (-0.08 to 0.29).

In [71]:
summary.columns = ["Annual Return", "Annual Volatility", "Sharpe Ratio"]

print(summary)

                  Annual Return  Annual Volatility  Sharpe Ratio
Ticker                                                          
Nasdaq-100             0.249745           0.182416      1.163524
Legal & General        0.099418           0.162597      0.380806
Semiconductors         0.452661           0.423406      0.980526
Gold                   0.253581           0.193206      1.118398
Global equity          0.180945           0.115741      1.239360
Emerging markets       0.213898           0.158859      1.110406
Portfolio              0.220062           0.136421      1.338221


In [ ]:
#Changing returns format to long so that it can be exported to excel

returns_long = returns.reset_index().melt(id_vars="Date", var_name="Holding", value_name="Return")
print(returns_long.head())
print(len(returns_long))


        Date     Holding    Return
0 2023-12-01  Nasdaq-100  0.066837
1 2024-01-01  Nasdaq-100  0.018512
2 2024-02-01  Nasdaq-100  0.041777
3 2024-03-01  Nasdaq-100  0.016610
4 2024-04-01  Nasdaq-100 -0.032446
204


In [78]:
with pd.ExcelWriter("portfolio_analysis.xlsx") as writer:
    summary.to_excel(writer, sheet_name="Summary")
    corr.to_excel(writer, sheet_name="Correlation")
    returns_long.to_excel(writer, sheet_name="Returns", index=False)
    # add corr and returns_long the same way

## Finding 4: Returns were concentrated in particular years
- From the Excel pivot table, Semiconductors lost money in 2024 (-0.4% a month on average), then averaged 9.0% a month in 2026.
- Gold did most of its work in 2025 (3.7% a month) and has been nearly flat in 2026 (0.1%).
- Legal & General averaged -0.6% a month in 2024.
- So the three-year averages make some holdings look steadier than they really were. 2023 is only one month and 2026 is nine, so those years are less reliable.